# 몬테카를로 실습

**Monte Carlo Sampling · MC · 준몬테카를로**

입력을 무작위로 반복 표본추출해 출력의 분포와 통계량을 추정하는 방법.

소재 분야에서 이해하기: 입력 오차 분포를 넣고 예측 물성의 분포를 얻는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SciPy 준몬테카를로 문서](https://docs.scipy.org/doc/scipy/reference/stats.qmc.html)

## 1. 입력 분포에서 출력 분포로

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 두께와 전도도 측정에 각각 오차가 있을 때, 계산된 저항의 분포를 봅니다.
thickness = rng.normal(500e-9, 25e-9, 200000)     # 500 nm ± 25 nm
conductivity = rng.normal(1.2e6, 6e4, 200000)     # S/m
length, width = 1e-3, 1e-4
resistance = length / (conductivity * thickness * width)
print('저항 평균 %.2f Ohm, 표준편차 %.2f Ohm' % (resistance.mean(), resistance.std()))
print('2.5-97.5 백분위 구간 %.2f - %.2f Ohm' % tuple(np.percentile(resistance, [2.5, 97.5])))
plt.hist(resistance, bins=80, density=True); plt.xlabel('resistance (Ohm)'); plt.show()

## 2. 표본 수와 정확도

몬테카를로 오차는 표본 수의 제곱근에 반비례합니다.

In [ ]:
truth = resistance.mean()
for size in (100, 1000, 10000, 100000):
    estimates = [np.mean(length / (rng.normal(1.2e6, 6e4, size) * rng.normal(500e-9, 25e-9, size) * width))
                 for _ in range(30)]
    print('표본 %6d -> 평균 추정의 표준편차 %.4f Ohm' % (size, np.std(estimates)))

## 3. 준몬테카를로가 더 빠를 수 있습니다

In [ ]:
from scipy.stats import qmc, norm

def integrate(sampler_points):
    t = norm.ppf(sampler_points[:, 0], 500e-9, 25e-9)
    c = norm.ppf(sampler_points[:, 1], 1.2e6, 6e4)
    return np.mean(length / (c * t * width))

for size in (64, 256, 1024, 4096):
    random_error = np.std([integrate(rng.random((size, 2))) for _ in range(20)])
    sobol_error = np.std([integrate(qmc.Sobol(2, scramble=True, seed=seed).random(size)) for seed in range(20)])
    print('표본 %5d -> 무작위 오차 %.5f / Sobol 수열 오차 %.5f' % (size, random_error, sobol_error))
print('\n같은 표본 수에서 준몬테카를로가 오차가 작으면 계산 비용을 절약할 수 있습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#monte-carlo)을 여세요.